In [ ]:
# -*- coding: utf-8 -*-

from pathlib import Path
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk

# 如缺依赖，先运行：
# !pip install SimpleITK pandas numpy openpyxl

# ============================================================
# 1. 配置
# ============================================================

def require_env_path(name: str) -> Path:
    """Return a required absolute path from an environment variable."""
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Set {name} to an absolute path before running this notebook."
        )
    path = Path(value).expanduser()
    if not path.is_absolute():
        raise RuntimeError(f"{name} must be an absolute path: {value!r}")
    return path.resolve()

PROJECT_DIR = require_env_path("NPC_PROJECT_ROOT")
DATA_ROOT = require_env_path("NPC_DATA_ROOT")

MASTER_XLSX = (
    PROJECT_DIR /
    "NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx"
)

OUTPUT_DIR = PROJECT_DIR / "patch_coverage_QC_final489"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_XLSX = OUTPUT_DIR / "patch_coverage_QC_489.xlsx"
REPORT_CSV = OUTPUT_DIR / "patch_coverage_QC_489.csv"
SUMMARY_TXT = OUTPUT_DIR / "patch_coverage_QC_summary.txt"

EXPECTED_TOTAL = 489

TARGET_SPACING_XYZ = (2.0, 2.0, 3.0)
PATCH_SIZE_XYZ = (80, 112, 64)
EXPECTED_ARRAY_SHAPE_ZYX = (64, 112, 80)

# 与旧正式预处理保持一致
ORAL_RETENTION_REQUIRED = 0.999
GTV_RETENTION_WARNING = 0.990

GTV_OVERRIDE_DIRS = [
    PROJECT_DIR / "GTV_inward_3mm" / "masks-gtv",
    PROJECT_DIR / "reconverted_masks" / "masks-gtv",
]

ORAL_OVERRIDE_DIRS = [
    PROJECT_DIR / "resegmented_masks" / "masks-oral",
    PROJECT_DIR / "oral_redraw_529_533" / "predictions",
]

FORBIDDEN_PATH_TOKENS = [
    "corrected_masks",
    "mask_direction_repair",
]


# ============================================================
# 2. 工具函数
# ============================================================

def norm_pid(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    s = str(v).strip()
    if s.endswith(".0"):
        s = s[:-2]
    try:
        return str(int(float(s)))
    except Exception:
        return s


def forbidden(path):
    t = str(path).lower()
    return any(x.lower() in t for x in FORBIDDEN_PATH_TOKENS)


def candidate_names(pid):
    pid = norm_pid(pid)
    return [
        f"{pid}.nii.gz",
        f"{pid}.nii",
        pid,
    ]


def resolve_path_value(value, pid):
    if value is None or (
        isinstance(value, float) and np.isnan(value)
    ):
        return None

    s = str(value).strip()

    if s.lower() in {
        "",
        "nan",
        "none",
        "null",
        "na",
        "n/a",
        "no_mask_gtv",
        "missing",
        "not_found",
    }:
        return None

    p = Path(s)

    if p.exists() and p.is_file() and not forbidden(p):
        return p

    if p.exists() and p.is_dir():
        for name in candidate_names(pid):
            q = p / name
            if q.exists() and q.is_file() and not forbidden(q):
                return q

    for suffix in [".nii.gz", ".nii"]:
        q = Path(s + suffix)
        if q.exists() and q.is_file() and not forbidden(q):
            return q

    return None


def find_in_dirs(pid, directories):
    for d in directories:
        d = Path(d)

        if not d.exists():
            continue

        for name in candidate_names(pid):
            p = d / name

            if (
                p.exists()
                and p.is_file()
                and not forbidden(p)
            ):
                return p

    return None


def choose_ct(pid, master_value):
    p = resolve_path_value(master_value, pid)

    if p is not None:
        return p, "master"

    p = find_in_dirs(
        pid,
        [DATA_ROOT / "ct"],
    )

    return (
        p,
        "fallback" if p is not None else "not_found",
    )


def choose_mask(pid, modality, master_value):

    if modality == "oral":

        p = find_in_dirs(
            pid,
            ORAL_OVERRIDE_DIRS,
        )

        if p is not None:
            return p, "project_final_override"

        p = resolve_path_value(
            master_value,
            pid,
        )

        if p is not None:
            return p, "master"

        p = find_in_dirs(
            pid,
            [DATA_ROOT / "masks-oral"],
        )

        return (
            p,
            "fallback" if p is not None else "not_found",
        )

    if modality == "gtv":

        p = find_in_dirs(
            pid,
            GTV_OVERRIDE_DIRS,
        )

        if p is not None:
            return p, "project_final_override"

        p = resolve_path_value(
            master_value,
            pid,
        )

        if p is not None:
            return p, "master"

        p = find_in_dirs(
            pid,
            [DATA_ROOT / "masks-gtv"],
        )

        return (
            p,
            "fallback" if p is not None else "not_found",
        )

    raise ValueError(modality)


def find_col(df, candidates):
    mp = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for name in candidates:
        if name.lower() in mp:
            return mp[name.lower()]

    return None


def binary_array(img):
    return (
        sitk.GetArrayFromImage(img).astype(np.float32)
        > 0.5
    ).astype(np.uint8)


def same_geometry(a, b, atol=1e-5):
    return (
        tuple(a.GetSize()) == tuple(b.GetSize())
        and np.allclose(
            a.GetSpacing(),
            b.GetSpacing(),
            atol=atol,
        )
        and np.allclose(
            a.GetOrigin(),
            b.GetOrigin(),
            atol=atol,
        )
        and np.allclose(
            a.GetDirection(),
            b.GetDirection(),
            atol=atol,
        )
    )


def oral_bbox_center(mask):
    arr = binary_array(mask)

    coords = np.argwhere(arr > 0)

    if coords.size == 0:
        raise ValueError("oral mask为空")

    min_zyx = coords.min(axis=0).astype(float)
    max_zyx = coords.max(axis=0).astype(float)

    center_zyx = (
        min_zyx + max_zyx
    ) / 2.0

    center_xyz = (
        float(center_zyx[2]),
        float(center_zyx[1]),
        float(center_zyx[0]),
    )

    center_physical = (
        mask.TransformContinuousIndexToPhysicalPoint(
            center_xyz
        )
    )

    return (
        tuple(float(x) for x in center_physical),
        center_xyz,
    )


def create_patch_reference(ct, center_physical):
    direction = np.asarray(
        ct.GetDirection(),
        dtype=float,
    ).reshape(3, 3)

    spacing = np.asarray(
        TARGET_SPACING_XYZ,
        dtype=float,
    )

    size = np.asarray(
        PATCH_SIZE_XYZ,
        dtype=float,
    )

    center_index = (
        size - 1.0
    ) / 2.0

    origin = (
        np.asarray(center_physical)
        - direction @ (
            center_index * spacing
        )
    )

    ref = sitk.Image(
        list(PATCH_SIZE_XYZ),
        sitk.sitkUInt8,
    )

    ref.SetSpacing(
        TARGET_SPACING_XYZ
    )

    ref.SetDirection(
        ct.GetDirection()
    )

    ref.SetOrigin(
        tuple(float(x) for x in origin)
    )

    return ref


def retained_fraction(mask, patch_reference):
    """
    将整个patch视野反向映射到原始mask网格，
    统计原mask体素中心落入patch的比例。

    这正是检测ROI是否被patch裁掉的指标。
    """

    patch_ones = sitk.GetImageFromArray(
        np.ones(
            EXPECTED_ARRAY_SHAPE_ZYX,
            dtype=np.uint8,
        )
    )

    patch_ones.CopyInformation(
        patch_reference
    )

    fov_on_original = sitk.Resample(
        patch_ones,
        mask,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        sitk.sitkUInt8,
    )

    original = binary_array(mask) > 0

    fov = (
        sitk.GetArrayFromImage(
            fov_on_original
        ) > 0
    )

    total = int(
        original.sum()
    )

    if total == 0:
        return np.nan, 0, 0

    inside = int(
        np.logical_and(
            original,
            fov,
        ).sum()
    )

    return (
        float(inside / total),
        inside,
        total,
    )


def volume_cc(mask):
    return float(
        binary_array(mask).sum()
        * np.prod(mask.GetSpacing())
        / 1000.0
    )


# ============================================================
# 3. 读取最终489例master
# ============================================================

if not MASTER_XLSX.exists():
    raise FileNotFoundError(
        f"未找到最终Master：\n{MASTER_XLSX}"
    )

df = pd.read_excel(
    MASTER_XLSX
)

PID_COL = find_col(
    df,
    [
        "patient_id",
        "patient",
        "case_id",
        "id",
    ],
)

CENTER_COL = find_col(
    df,
    [
        "model_center",
        "center",
    ],
)

LABEL_COL = find_col(
    df,
    [
        "severe_mucositis",
        "label",
        "outcome",
    ],
)

CT_COL = find_col(
    df,
    [
        "ct_path",
        "planning_ct_path",
        "ct_nifti_path",
        "ct",
    ],
)

ORAL_COL = find_col(
    df,
    [
        "oral_cavity_mask_path",
        "oral_mask_path",
        "oral_path",
        "mask_oral_path",
    ],
)

GTV_COL = find_col(
    df,
    [
        "gtv_mask_path",
        "gtv_path",
        "gtvnx_mask_path",
        "gtv_nx_mask_path",
        "mask_gtv_path",
    ],
)

if PID_COL is None:
    raise RuntimeError(
        "无法识别patient_id列。\n"
        f"现有列：{list(df.columns)}"
    )

# ============================================================
# 根据最终Frozen Master的exclude_reason筛选正式489例
# ============================================================

df["_pid"] = df[PID_COL].map(norm_pid)

# 先保留有patient_id的全部assessed病例
df = (
    df[df["_pid"] != ""]
    .drop_duplicates("_pid", keep="first")
    .reset_index(drop=True)
)

print(f"Master assessed patients = {len(df)}")

if len(df) != 540:
    raise RuntimeError(
        f"Frozen Master应包含540例assessed patients，"
        f"当前读取到{len(df)}例。"
    )

# 必须存在最终排除原因字段
if "exclude_reason" not in df.columns:
    raise RuntimeError(
        "Master中未找到 exclude_reason 列。"
        "请不要自行根据影像缺失重新定义队列。"
    )

# exclude_reason为空 = 最终纳入
exclude_text = (
    df["exclude_reason"]
    .fillna("")
    .astype(str)
    .str.strip()
)

excluded_df = df.loc[
    ~exclude_text.eq("")
].copy()

df = df.loc[
    exclude_text.eq("")
].copy()

df = df.reset_index(drop=True)

print()
print("=" * 90)
print("=" * 90)

print(f"Assessed = {len(df) + len(excluded_df)}")
print(f"Excluded = {len(excluded_df)}")
print(f"Included = {len(df)}")

print()
print("Exclude reasons:")
print(
    excluded_df["exclude_reason"]
    .fillna("")
    .astype(str)
    .str.strip()
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 强制验证最终正式队列
# ------------------------------------------------------------

if len(excluded_df) != 51:
    raise RuntimeError(
        f"最终应排除51例，当前识别到{len(excluded_df)}例。"
    )

if len(df) != EXPECTED_TOTAL:
    raise RuntimeError(
        f"最终正式分析队列应为{EXPECTED_TOTAL}例，"
        f"当前筛选后为{len(df)}例。"
    )

# ------------------------------------------------------------
# 验证model_center
# 注意：正式队列定义必须使用model_center，不使用旧cohort字段
# ------------------------------------------------------------

if CENTER_COL is None:
    raise RuntimeError(
        "未找到model_center列，无法验证最终中心构成。"
    )

df[CENTER_COL] = (
    df[CENTER_COL]
    .astype(str)
    .str.strip()
    .str.upper()
)

center_counts = (
    df[CENTER_COL]
    .value_counts()
    .to_dict()
)

print()
print(center_counts)

EXPECTED_CENTER_COUNTS = {
    "A": 180,
    "B": 81,
    "C": 228,
}

if center_counts != EXPECTED_CENTER_COUNTS:
    raise RuntimeError(
        "最终中心人数与Frozen设计不一致。\n"
        f"Expected: {EXPECTED_CENTER_COUNTS}\n"
        f"Observed: {center_counts}"
    )

# ------------------------------------------------------------
# 验证最终标签
# ------------------------------------------------------------

if LABEL_COL is not None:

    labels = pd.to_numeric(
        df[LABEL_COL],
        errors="coerce",
    )

    if labels.isna().any():
        bad_ids = df.loc[
            labels.isna(),
            "_pid",
        ].tolist()

        raise RuntimeError(
            "最终489例存在缺失severe_mucositis标签："
            f"{bad_ids}"
        )

    label_counts = (
        labels.astype(int)
        .value_counts()
        .sort_index()
        .to_dict()
    )

    print()
    print(label_counts)

    EXPECTED_LABEL_COUNTS = {
        0: 246,
        1: 243,
    }

    if label_counts != EXPECTED_LABEL_COUNTS:
        raise RuntimeError(
            "最终标签人数与正式489例不一致。\n"
            f"Expected: {EXPECTED_LABEL_COUNTS}\n"
            f"Observed: {label_counts}"
        )

print()
print("Development = B + C = 309")
print("External    = A     = 180")
print("=" * 90)

print("=" * 90)
print("=" * 90)

print("Master:", MASTER_XLSX)
print("N =", len(df))

print(
    "Detected columns:",
    {
        "PID": PID_COL,
        "CENTER": CENTER_COL,
        "LABEL": LABEL_COL,
        "CT": CT_COL,
        "ORAL": ORAL_COL,
        "GTV": GTV_COL,
    },
)


# ============================================================
# 4. 逐例计算
# ============================================================

records = []

for i, row in df.iterrows():

    pid = row["_pid"]

    rec = {
        "patient_id": pid,
        "model_center":
            row.get(
                CENTER_COL,
                "",
            )
            if CENTER_COL
            else "",
        "severe_mucositis":
            row.get(
                LABEL_COL,
                np.nan,
            )
            if LABEL_COL
            else np.nan,
        "status": "PENDING",
        "error": "",
    }

    try:

        ct_path, ct_source = choose_ct(
            pid,
            row.get(CT_COL)
            if CT_COL
            else None,
        )

        oral_path, oral_source = (
            choose_mask(
                pid,
                "oral",
                row.get(ORAL_COL)
                if ORAL_COL
                else None,
            )
        )

        gtv_path, gtv_source = (
            choose_mask(
                pid,
                "gtv",
                row.get(GTV_COL)
                if GTV_COL
                else None,
            )
        )

        rec.update({
            "ct_path":
                str(ct_path)
                if ct_path
                else "",
            "ct_path_source":
                ct_source,
            "oral_path":
                str(oral_path)
                if oral_path
                else "",
            "oral_path_source":
                oral_source,
            "gtv_path":
                str(gtv_path)
                if gtv_path
                else "",
            "gtv_path_source":
                gtv_source,
        })

        if (
            ct_path is None
            or oral_path is None
            or gtv_path is None
        ):
            raise FileNotFoundError(
                "CT/oral/GTV路径缺失"
            )

        ct = sitk.ReadImage(
            str(ct_path)
        )

        oral = sitk.Cast(
            sitk.ReadImage(
                str(oral_path)
            ) > 0,
            sitk.sitkUInt8,
        )

        gtv = sitk.Cast(
            sitk.ReadImage(
                str(gtv_path)
            ) > 0,
            sitk.sitkUInt8,
        )

        oral_same = same_geometry(
            ct,
            oral,
        )

        gtv_same = same_geometry(
            ct,
            gtv,
        )

        rec[
            "oral_same_geometry_as_CT"
        ] = oral_same

        rec[
            "gtv_same_geometry_as_CT"
        ] = gtv_same

        if (
            not oral_same
            or not gtv_same
        ):
            raise RuntimeError(
                "进入patch QC前geometry不一致："
                f"oral={oral_same}, "
                f"GTV={gtv_same}。"
                "请核实正式预处理实际使用的mask。"
            )

        (
            oral_center_physical,
            oral_center_index,
        ) = oral_bbox_center(
            oral
        )

        patch_ref = (
            create_patch_reference(
                ct,
                oral_center_physical,
            )
        )

        (
            oral_ret,
            oral_inside,
            oral_total,
        ) = retained_fraction(
            oral,
            patch_ref,
        )

        (
            gtv_ret,
            gtv_inside,
            gtv_total,
        ) = retained_fraction(
            gtv,
            patch_ref,
        )

        rec.update({

            "oral_retained_fraction":
                oral_ret,

            "oral_retained_percent":
                oral_ret * 100,

            "oral_voxels_total":
                oral_total,

            "oral_voxels_inside_patch":
                oral_inside,

            "oral_voxels_clipped":
                oral_total
                - oral_inside,

            "oral_volume_cc":
                volume_cc(oral),

            "oral_pass_ge_0.999":
                bool(
                    oral_ret
                    >= ORAL_RETENTION_REQUIRED
                ),

            "gtv_retained_fraction":
                gtv_ret,

            "gtv_retained_percent":
                gtv_ret * 100,

            "gtv_voxels_total":
                gtv_total,

            "gtv_voxels_inside_patch":
                gtv_inside,

            "gtv_voxels_clipped":
                gtv_total
                - gtv_inside,

            "gtv_volume_cc":
                volume_cc(gtv),

            "gtv_pass_ge_0.99":
                bool(
                    gtv_ret
                    >= GTV_RETENTION_WARNING
                ),

            "oral_center_physical_x_mm":
                oral_center_physical[0],

            "oral_center_physical_y_mm":
                oral_center_physical[1],

            "oral_center_physical_z_mm":
                oral_center_physical[2],

            "patch_origin_x_mm":
                patch_ref.GetOrigin()[0],

            "patch_origin_y_mm":
                patch_ref.GetOrigin()[1],

            "patch_origin_z_mm":
                patch_ref.GetOrigin()[2],
        })

        warnings_case = []

        if (
            oral_ret
            < ORAL_RETENTION_REQUIRED
        ):
            warnings_case.append(
                f"oral={oral_ret:.6f}"
            )

        if (
            gtv_ret
            < GTV_RETENTION_WARNING
        ):
            warnings_case.append(
                f"gtv={gtv_ret:.6f}"
            )

        if warnings_case:
            rec["status"] = "WARNING"
            rec["warning"] = "; ".join(
                warnings_case
            )
        else:
            rec["status"] = "PASS"
            rec["warning"] = ""

    except Exception as e:

        rec["status"] = "ERROR"

        rec["error"] = (
            f"{type(e).__name__}: {e}"
        )

    records.append(rec)

    if (
        (i + 1) % 25 == 0
        or i + 1 == len(df)
    ):
        print(
            f"Processed "
            f"{i + 1}/{len(df)}"
        )


# ============================================================
# 5. 汇总
# ============================================================

result_df = pd.DataFrame(
    records
)

valid_df = result_df[
    result_df["status"].isin(
        ["PASS", "WARNING"]
    )
].copy()

warning_df = result_df[
    result_df["status"] == "WARNING"
].copy()

error_df = result_df[
    result_df["status"] == "ERROR"
].copy()

summary = [
    [
        "expected_total",
        EXPECTED_TOTAL,
    ],
    [
        "processed_total",
        len(result_df),
    ],
    [
        "pass_no_warning",
        int(
            (
                result_df["status"]
                == "PASS"
            ).sum()
        ),
    ],
    [
        "warning_n",
        int(
            (
                result_df["status"]
                == "WARNING"
            ).sum()
        ),
    ],
    [
        "error_n",
        int(
            (
                result_df["status"]
                == "ERROR"
            ).sum()
        ),
    ],
]

if len(valid_df):

    summary += [

        [
            "oral_ge_0.999_n",
            int(
                (
                    valid_df[
                        "oral_retained_fraction"
                    ] >= 0.999
                ).sum()
            ),
        ],

        [
            "oral_lt_0.999_n",
            int(
                (
                    valid_df[
                        "oral_retained_fraction"
                    ] < 0.999
                ).sum()
            ),
        ],

        [
            "oral_retention_min",
            float(
                valid_df[
                    "oral_retained_fraction"
                ].min()
            ),
        ],

        [
            "oral_retention_median",
            float(
                valid_df[
                    "oral_retained_fraction"
                ].median()
            ),
        ],

        [
            "gtv_ge_0.99_n",
            int(
                (
                    valid_df[
                        "gtv_retained_fraction"
                    ] >= 0.99
                ).sum()
            ),
        ],

        [
            "gtv_0.95_to_0.99_n",
            int(
                (
                    (
                        valid_df[
                            "gtv_retained_fraction"
                        ] >= 0.95
                    )
                    &
                    (
                        valid_df[
                            "gtv_retained_fraction"
                        ] < 0.99
                    )
                ).sum()
            ),
        ],

        [
            "gtv_lt_0.95_n",
            int(
                (
                    valid_df[
                        "gtv_retained_fraction"
                    ] < 0.95
                ).sum()
            ),
        ],

        [
            "gtv_retention_min",
            float(
                valid_df[
                    "gtv_retained_fraction"
                ].min()
            ),
        ],

        [
            "gtv_retention_median",
            float(
                valid_df[
                    "gtv_retained_fraction"
                ].median()
            ),
        ],
    ]

summary_df = pd.DataFrame(
    summary,
    columns=["item", "value"],
)


# ============================================================
# 6. 保存结果
# ============================================================

result_df.to_csv(
    REPORT_CSV,
    index=False,
    encoding="utf-8-sig",
)

with pd.ExcelWriter(
    REPORT_XLSX,
    engine="openpyxl",
) as writer:

    result_df.to_excel(
        writer,
        sheet_name="Case_level",
        index=False,
    )

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False,
    )

    warning_df.to_excel(
        writer,
        sheet_name="Warnings",
        index=False,
    )

    error_df.to_excel(
        writer,
        sheet_name="Errors",
        index=False,
    )

SUMMARY_TXT.write_text(
    summary_df.to_string(
        index=False
    ),
    encoding="utf-8",
)


# ============================================================
# 7. 显示最终结果
# ============================================================

print()
print("=" * 90)
print("QC SUMMARY")
print("=" * 90)

print(
    summary_df.to_string(
        index=False
    )
)

print()
print("Excel:", REPORT_XLSX)
print("CSV  :", REPORT_CSV)
print("TXT  :", SUMMARY_TXT)

if len(warning_df):

    print()
    print("WARNING CASES")

    show_cols = [
        "patient_id",
        "model_center",
        "oral_retained_fraction",
        "gtv_retained_fraction",
        "warning",
    ]

    print(
        warning_df[
            show_cols
        ]
        .sort_values(
            "gtv_retained_fraction"
        )
        .to_string(
            index=False
        )
    )

if len(error_df):

    print()
    print("ERROR CASES")

    print(
        error_df[
            [
                "patient_id",
                "model_center",
                "error",
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# 8. 自动判读
# ============================================================

if len(error_df) > 0:

    print()
    print(
        "⚠ 存在ERROR病例。"
        "在ERROR清零前不要据此修改论文。"
    )

else:

    oral_bad = int(
        (
            result_df[
                "oral_retained_fraction"
            ] < 0.999
        ).sum()
    )

    gtv_warn = int(
        (
            result_df[
                "gtv_retained_fraction"
            ] < 0.99
        ).sum()
    )

    gtv_major = int(
        (
            result_df[
                "gtv_retained_fraction"
            ] < 0.95
        ).sum()
    )

    print()
    print("AUTO INTERPRETATION")

    if (
        oral_bad == 0
        and gtv_warn == 0
    ):

        print(
            "✅ 489例oral和GTV均达到"
            "预设patch coverage标准。"
        )

    elif (
        oral_bad == 0
        and gtv_major == 0
    ):

        print(
            f"🟡 oral全部达标；"
            f"{gtv_warn}例GTV低于99%，"
            "但无GTV低于95%。"
            "建议人工复核这些warning病例。"
        )

    else:

        print(
            f"⚠ oral <99.9%："
            f"{oral_bad}例；"
            f"GTV <99%："
            f"{gtv_warn}例；"
            f"GTV <95%："
            f"{gtv_major}例。"
        )

        print(
            "建议人工复核明显截断病例，"
            "再决定是否需要敏感性分析。"
        )